# Notebook 3: Entity Linking and Knowledge Representation

This notebook links extracted claim subjects and objects to Wikidata entities. The output is a structured JSON file for knowledge graph reasoning.


In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm
from google.colab import files

In [2]:
uploaded = files.upload()

INPUT_PATH = Path("/content/extracted_triples_filtered.json")
OUTPUT_PATH = Path("/content/linked_entities.json")

if not INPUT_PATH.exists():
    uploaded_name = list(uploaded.keys())[0]
    INPUT_PATH = Path("/content") / uploaded_name

print(f"Using input file: {INPUT_PATH}")

Saving extracted_triples_filtered.json to extracted_triples_filtered (2).json
Using input file: /content/extracted_triples_filtered.json


In [3]:
# Load extracted triples

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    triples = json.load(f)

df = pd.DataFrame(triples)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (500, 11)


,raw_claim,speaker,label,subject,subject_type,relation,object,object_type,date,confidence,entities
0,Says the Annies List political group supports ...,dwayne-bohac,false,Annies List,ORG,say,demand,UNKNOWN,None,0.8949,"[{""word"": "" Annies List"", ""type"": ""ORG"", ""scor..."
1,When did the decline of coal start? It started...,scott-surovell,half-true,the decline,UNKNOWN,do,George W.) Bush,PER,None,0.8983,"[{""word"": "" George W.) Bush"", ""type"": ""PER"", ""..."
2,"Hillary Clinton agrees with John McCain ""by vo...",barack-obama,mostly-true,Hillary Clinton,PER,agree,John McCain,PER,None,0.9998,"[{""word"": "" Hillary Clinton"", ""type"": ""PER"", ""..."
3,Health care reform legislation is likely to ma...,blog-posting,false,Health care reform legislation,UNKNOWN,be,free sex change surgeries,UNKNOWN,None,0.3000,[]
4,The economic turnaround started at the end of ...,charlie-crist,half-true,The economic turnaround,UNKNOWN,start,at the end,UNKNOWN,None,0.3000,[]


In [4]:
# Normalize Notebook 1 fields

if "claim_id" not in df.columns:
    df["claim_id"] = [f"claim-{i+1:05d}" for i in range(len(df))]

if "extraction_confidence" not in df.columns and "confidence" in df.columns:
    df["extraction_confidence"] = df["confidence"]

required_cols = [
    "claim_id",
    "raw_claim",
    "label",
    "speaker",
    "subject",
    "subject_type",
    "relation",
    "object",
    "object_type",
    "date",
    "extraction_confidence",
]

for col in required_cols:
    if col not in df.columns:
        df[col] = None

df = df[required_cols]

df.head()

,claim_id,raw_claim,label,speaker,subject,subject_type,relation,object,object_type,date,extraction_confidence
0,claim-00001,Says the Annies List political group supports ...,false,dwayne-bohac,Annies List,ORG,say,demand,UNKNOWN,None,0.8949
1,claim-00002,When did the decline of coal start? It started...,half-true,scott-surovell,the decline,UNKNOWN,do,George W.) Bush,PER,None,0.8983
2,claim-00003,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true,barack-obama,Hillary Clinton,PER,agree,John McCain,PER,None,0.9998
3,claim-00004,Health care reform legislation is likely to ma...,false,blog-posting,Health care reform legislation,UNKNOWN,be,free sex change surgeries,UNKNOWN,None,0.3000
4,claim-00005,The economic turnaround started at the end of ...,half-true,charlie-crist,The economic turnaround,UNKNOWN,start,at the end,UNKNOWN,None,0.3000


In [5]:
import requests

headers = {
    "User-Agent": "AIaP-Entity-Linking-Student-Project/1.0 (student project)"
}

params = {
    "action": "wbsearchentities",
    "format": "json",
    "language": "en",
    "search": "Hillary Clinton",
    "limit": 1,
}

response = requests.get(
    "https://www.wikidata.org/w/api.php",
    params=params,
    headers=headers,
    timeout=10,
)

print("Status:", response.status_code)
print(response.text[:1000])


Status: 200
{"searchinfo":{"search":"Hillary Clinton"},"search":[{"id":"Q6294","title":"Q6294","pageid":7407,"concepturi":"http://www.wikidata.org/entity/Q6294","repository":"wikidata","url":"//www.wikidata.org/wiki/Q6294","display":{"label":{"value":"Hillary Clinton","language":"en"},"description":{"value":"American politician and diplomat (born 1947)","language":"en"}},"label":"Hillary Clinton","description":"American politician and diplomat (born 1947)","match":{"type":"label","language":"en","text":"Hillary Clinton"}}],"search-continue":1,"success":1}


In [6]:
# Relation-to-Wikidata-property mapping

PROPERTY_MAP = {
    "founded_in": "P571",
    "founded": "P571",
    "born_in": "P19",
    "place_of_birth": "P19",
    "died_in": "P20",
    "place_of_death": "P20",
    "position_held": "P39",
    "president_of": "P39",
    "located_in": "P131",
    "part_of": "P361",
    "member_of": "P463",
    "citizen_of": "P27",
    "country": "P17",
    "educated_at": "P69",
    "spouse": "P26",
    "capital_of": "P1376",
    "population": "P1082",
    "owned_by": "P127",
    "headquarters": "P159",
    "leader": "P6",
    "say": None,
    "says": None,
    "said": None,
    "claim": None,
    "claims": None,
    "agree": None,
    "agrees": None,
    "support": None,
    "supports": None,
    "oppose": None,
    "opposes": None,
}


## Method

The notebook links only entity-like values such as people, organisations, locations, and miscellaneous named entities. Vague phrases and literals are skipped to reduce false matches.


In [7]:
# Helper functions

import time
import requests

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

HEADERS = {
    "User-Agent": "AIaP-Entity-Linking-Student-Project/1.0 (student project)"
}

entity_cache = {}

LINKABLE_TYPES = {"PER", "PERSON", "ORG", "LOC", "GPE", "MISC"}

def clean_text(value):
    if value is None:
        return None

    value = str(value).strip()

    if not value or value.lower() in {"none", "nan", "unknown", "null"}:
        return None

    return value


def should_link_entity(value, entity_type):
    value = clean_text(value)

    if value is None:
        return False

    if entity_type not in LINKABLE_TYPES:
        return False

    bad_values = {"it", "we", "they", "you", "i", "he", "she", "one", "most", "much"}
    if value.lower() in bad_values:
        return False

    if len(value) <= 1:
        return False

    return True


def is_literal(value, entity_type=None):
    value = clean_text(value)

    if value is None:
        return True

    literal_types = {
        "DATE", "TIME", "CARDINAL", "PERCENT",
        "MONEY", "QUANTITY", "ORDINAL", "NUMBER"
    }

    if entity_type in literal_types:
        return True

    numeric_candidate = value.replace(",", "").replace(".", "").replace("%", "")
    if numeric_candidate.isdigit():
        return True

    return False


def normalize_relation(relation):
    relation = clean_text(relation)

    if relation is None:
        return None

    return relation.lower().strip().replace(" ", "_").replace("-", "_")


def map_relation_to_property(relation):
    normalized = normalize_relation(relation)

    if normalized is None:
        return None

    return PROPERTY_MAP.get(normalized)


def search_wikidata_entity(query, limit=1, sleep_seconds=1.0):
    query = clean_text(query)

    if query is None:
        return None

    cache_key = query.lower()

    if cache_key in entity_cache:
        return entity_cache[cache_key]

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "search": query,
        "limit": limit,
    }

    try:
        time.sleep(sleep_seconds)

        response = requests.get(
            WIKIDATA_API,
            params=params,
            headers=HEADERS,
            timeout=15,
        )

        if response.status_code == 429:
            print(f"Rate limited on: {query}. Waiting 10 seconds...")
            time.sleep(10)
            response = requests.get(
                WIKIDATA_API,
                params=params,
                headers=HEADERS,
                timeout=15,
            )

        if response.status_code != 200:
            entity_cache[cache_key] = None
            return None

        results = response.json().get("search", [])

        if not results:
            entity_cache[cache_key] = None
            return None

        top = results[0]

        result = {
            "kg_id": top.get("id"),
            "label": top.get("label"),
            "description": top.get("description"),
        }

        entity_cache[cache_key] = result
        return result

    except Exception as e:
        print(f"Failed query: {query} | {e}")
        entity_cache[cache_key] = None
        return None


In [8]:
print(search_wikidata_entity("Hillary Clinton"))
print(search_wikidata_entity("John McCain"))
print(search_wikidata_entity("Chicago Bears"))

{'kg_id': 'Q6294', 'label': 'Hillary Clinton', 'description': 'American politician and diplomat (born 1947)'}
{'kg_id': 'Q10390', 'label': 'John McCain', 'description': 'American politician (1936–2018)'}
{'kg_id': 'Q205033', 'label': 'Chicago Bears', 'description': 'American football team'}


In [13]:
# Link one claim

def calculate_linking_confidence(
    subject_link,
    object_link,
    object_is_literal,
    property_id,
    extraction_confidence,
):
    try:
        extraction_confidence = float(extraction_confidence)
    except Exception:
        extraction_confidence = 0.5

    subject_score = 1.0 if subject_link else 0.0

    if object_is_literal:
        object_score = 0.8
    else:
        object_score = 1.0 if object_link else 0.0

    property_score = 1.0 if property_id else 0.0

    confidence = (
        0.35 * subject_score
        + 0.25 * object_score
        + 0.15 * property_score
        + 0.25 * extraction_confidence
    )

    return round(confidence, 3)


def build_linking_notes(subject_link, object_link, object_is_literal, property_id):
    notes = []

    if subject_link:
        notes.append("Subject linked to Wikidata.")
    else:
        notes.append("Subject not linked or not suitable for linking.")

    if object_is_literal:
        notes.append("Object treated as literal value.")
    elif object_link:
        notes.append("Object linked to Wikidata.")
    else:
        notes.append("Object not linked or not suitable for linking.")

    if property_id:
        notes.append("Relation mapped to Wikidata property.")
    else:
        notes.append("No reliable Wikidata property mapping for relation.")

    return " ".join(notes)


def link_claim(row):
    subject = clean_text(row.get("subject"))
    obj = clean_text(row.get("object"))

    subject_type = row.get("subject_type")
    object_type = row.get("object_type")

    if should_link_entity(subject, subject_type):
        subject_link = search_wikidata_entity(subject)
    else:
        subject_link = None

    object_is_literal = is_literal(obj, object_type)

    if should_link_entity(obj, object_type) and not object_is_literal:
        object_link = search_wikidata_entity(obj)
    else:
        object_link = None

    property_id = map_relation_to_property(row.get("relation"))

    linking_confidence = calculate_linking_confidence(
        subject_link=subject_link,
        object_link=object_link,
        object_is_literal=object_is_literal,
        property_id=property_id,
        extraction_confidence=row.get("extraction_confidence"),
    )

    return {
        "claim_id": row.get("claim_id"),
        "raw_claim": row.get("raw_claim"),
        "label": row.get("label"),
        "speaker": row.get("speaker"),
        "subject": subject,
        "subject_type": subject_type,
        "subject_kg_id": subject_link["kg_id"] if subject_link else None,
        "subject_kg_label": subject_link["label"] if subject_link else None,
        "subject_kg_description": subject_link["description"] if subject_link else None,
        "relation": row.get("relation"),
        "property_id": property_id,
        "object": obj,
        "object_type": object_type,
        "object_kg_id": object_link["kg_id"] if object_link else None,
        "object_kg_label": object_link["label"] if object_link else None,
        "object_kg_description": object_link["description"] if object_link else None,
        "date": row.get("date"),
        "extraction_confidence": row.get("extraction_confidence"),
        "linking_confidence": linking_confidence,
        "linking_notes": build_linking_notes(
            subject_link,
            object_link,
            object_is_literal,
            property_id,
        ),
    }


In [14]:
link_claim(df.iloc[2])


{'claim_id': 'claim-00003',
 'raw_claim': 'Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on Iran."',
 'label': 'mostly-true',
 'speaker': 'barack-obama',
 'subject': 'Hillary Clinton',
 'subject_type': 'PER',
 'subject_kg_id': 'Q6294',
 'subject_kg_label': 'Hillary Clinton',
 'subject_kg_description': 'American politician and diplomat (born 1947)',
 'relation': 'agree',
 'property_id': None,
 'object': 'John McCain',
 'object_type': 'PER',
 'object_kg_id': 'Q10390',
 'object_kg_label': 'John McCain',
 'object_kg_description': 'American politician (1936–2018)',
 'date': None,
 'extraction_confidence': np.float64(0.9998000264),
 'linking_confidence': 0.85,
 'linking_notes': 'Subject linked to Wikidata. Object linked to Wikidata. No reliable Wikidata property mapping for relation.'}

## Results

The notebook processed 500 extracted claim triples. 207 subjects and 69 objects were linked to Wikidata entities.


In [22]:
# Run entity linking

entity_cache = {}

MAX_RECORDS = None
working_df = df.head(MAX_RECORDS)

linked_records = []

for _, row in tqdm(working_df.iterrows(), total=len(working_df)):
    linked_records.append(link_claim(row))

linked_df = pd.DataFrame(linked_records)

summary = {
    "total_claims": len(linked_df),
    "subjects_linked": int(linked_df["subject_kg_id"].notna().sum()),
    "objects_linked": int(linked_df["object_kg_id"].notna().sum()),
    "relations_with_property_id": int(linked_df["property_id"].notna().sum()),
    "average_linking_confidence": round(float(linked_df["linking_confidence"].mean()), 3),
}

summary



  4%|▍         | 20/500 [00:15<04:52,  1.64it/s]

Rate limited on: United States. Waiting 10 seconds...


  4%|▍         | 22/500 [00:26<17:15,  2.17s/it]

Rate limited on: Scott Walker. Waiting 10 seconds...


  5%|▍         | 24/500 [00:37<25:36,  3.23s/it]

...

Rate limited on: House. Waiting 10 seconds...


 94%|█████████▍| 469/500 [16:54<01:46,  3.44s/it]

Rate limited on: Flint. Waiting 10 seconds...


100%|██████████| 500/500 [17:14<00:00,  2.07s/it]


{'total_claims': 500,
 'subjects_linked': 207,
 'objects_linked': 69,
 'relations_with_property_id': 0,
 'average_linking_confidence': 0.377}

In [23]:
# Save linked output

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(linked_records, f, indent=2, ensure_ascii=False)

print(f"Saved output to {OUTPUT_PATH}")


Saved output to /content/linked_entities.json


In [24]:
# Download linked_entities.json

files.download(str(OUTPUT_PATH))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>